In [2]:
import os
import glob
import pandas as pd
import numpy as np
import joblib

from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix

# ---------------------------
# Load processed parquet data
# ---------------------------

def load_parquet_data(folder_path, sample_frac=0.2):
    files = glob.glob(f"{folder_path}/*.parquet")
    df_list = []

    for f in files:
        df = pd.read_parquet(f)
        df = df.sample(frac=sample_frac, random_state=42)  # take only 20%
        df_list.append(df)

    return pd.concat(df_list, ignore_index=True)

# ---------------------------
# Isolation Forest Model
# ---------------------------
def run_isolation_forest(train_df, test_df):
    X_train = train_df.drop(columns=["binary_label"])
    X_test = test_df.drop(columns=["binary_label"])
    y_test = test_df["binary_label"]

    model = IsolationForest(
        n_estimators=200,
        contamination=0.1,
        random_state=42,
        n_jobs=-1
    )

    print("Training Isolation Forest...")
    model.fit(X_train)

    joblib.dump(model, "models/isolation_forest.pkl")

    preds = model.predict(X_test)
    preds = np.where(preds == -1, 1, 0)

    print("\nIsolation Forest Results")
    print(confusion_matrix(y_test, preds))
    print(classification_report(y_test, preds))

# ---------------------------
# Main function
# ---------------------------
def main():
    print("Loading processed data...")

    train_df = load_parquet_data("data/processed/train")
    test_df = load_parquet_data("data/processed/test")

    print("Running anomaly detection...")
    run_isolation_forest(train_df, test_df)

    print("\nAnomaly detection completed successfully.")

if __name__ == "__main__":
    main()

Loading processed data...


ValueError: No objects to concatenate